In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier

import re
import spacy
import torch
import wikipedia
import trafilatura
from ddgs import DDGS
from sentence_transformers import CrossEncoder
from transformers import pipeline
import logging
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()


%matplotlib inline
sns.set_theme(style="whitegrid")
from scipy.stats import mode as scipy_mode
from collections import Counter


### 1. Load and Preprocess Data
Instead of concatenating `claim` and `evidence` into one string, we keep them as two
separate text fields. Each will get its own TF-IDF representation later, so the model
can weigh claim-vocabulary and evidence-vocabulary independently rather than blending
them into a single bag-of-words.

In [ ]:
# Load the dataset generated from the previous pipeline
df = pd.read_json("dataset.jsonl", lines=True)

# Handle any missing values
df['claim'] = df['claim'].fillna("None")
df['evidence'] = df['evidence'].fillna("None")

print(f"Total records loaded: {len(df)}")
print("\nLabel Distribution:")
print(df['label'].value_counts())

In [ ]:
from scipy.sparse import hstack
X_claim = df['claim']
X_evidence = df['evidence']
y = df['label']

(X_claim_train, X_claim_test,
 X_evidence_train, X_evidence_test,
 y_train, y_test) = train_test_split(
    X_claim, X_evidence, y, test_size=0.2, random_state=42, stratify=y
)

claim_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 3), min_df=3, max_df=0.8, sublinear_tf=True)
evidence_vectorizer = TfidfVectorizer(max_features=100000, ngram_range=(1, 3), min_df=3, max_df=0.8, sublinear_tf=True)

X_claim_train_vec = claim_vectorizer.fit_transform(X_claim_train)
X_claim_test_vec = claim_vectorizer.transform(X_claim_test)

X_evidence_train_vec = evidence_vectorizer.fit_transform(X_evidence_train)
X_evidence_test_vec = evidence_vectorizer.transform(X_evidence_test)

X_train_vec = hstack([X_claim_train_vec, X_evidence_train_vec]).tocsr()
X_test_vec = hstack([X_claim_test_vec, X_evidence_test_vec]).tocsr()

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(C=5.0, max_iter=2000, random_state=42),
    "Linear SVC": LinearSVC(C=0.5, max_iter=2000, random_state=42, dual=False, class_weight='balanced'),
    "Multinomial NB": MultinomialNB(alpha=0.1),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=100),
    "Random Forest": RandomForestClassifier(n_estimators=300,max_depth=50, min_samples_leaf=2,max_features=0.1, random_state=42, n_jobs=-1),
    "Gradient Boosting": LGBMClassifier(n_estimators=500,learning_rate=0.05,num_leaves=63,subsample=0.8,colsample_bytree=0.8,random_state=42,n_jobs=-1),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(256, 64),activation="relu",solver="adam",alpha=1e-4,batch_size=256,learning_rate="adaptive",max_iter=20,early_stopping=True,n_iter_no_change=5,random_state=42)
}

### 4. Training and Evaluation Pipeline

In [ ]:
import copy
def fit_mlp_checkpointed(model, X, y, ckpt_path, save_every=1, val_fraction=0.1):
    """Train an MLPClassifier epoch by epoch, saving a checkpoint every `save_every`
    epochs. If ckpt_path exists, training resumes from it (weights, Adam state, early-stopping state)."""
    # partial_fit doesn't support early_stopping=True, so we replicate it manually
    X_tr, X_val, y_tr, y_val = train_test_split(
        X, y, test_size=val_fraction, stratify=y, random_state=42)  # deterministic -> same split on resume
    if os.path.exists(ckpt_path):
        ck = joblib.load(ckpt_path)
        model, st = ck['model'], ck['state']
        print(f"  Resuming MLP from epoch {st['epoch']}")
    else:
        model.set_params(early_stopping=False)
        st = dict(epoch=0, best=-np.inf, best_w=None, bad=0, elapsed=0.0)

    while st['epoch'] < model.max_iter and st['bad'] <= model.n_iter_no_change:
        t0 = time.time()
        model.partial_fit(X_tr, y_tr, classes=np.unique(y))
        score = model.score(X_val, y_val)
        st['bad'] = st['bad'] + 1 if score < st['best'] + model.tol else 0
        if score > st['best']:
            st['best'] = score
            st['best_w'] = copy.deepcopy((model.coefs_, model.intercepts_))
        st['epoch'] += 1
        st['elapsed'] += time.time() - t0
        print(f"  Epoch {st['epoch']}: loss={model.loss_:.4f} val_acc={score:.4f}")
        if st['epoch'] % save_every == 0:
            joblib.dump({'model': model, 'state': st}, ckpt_path + ".tmp")
            os.replace(ckpt_path + ".tmp", ckpt_path)   # atomic: never leaves a corrupt checkpoint

    model.coefs_, model.intercepts_ = st['best_w']      # restore best weights (as early_stopping=True does)
    return model, st['elapsed']

results = []
for name, model in models.items():
    # Create a safe filename for the model
    safe_name = name.replace(" ", "_").replace("(", "").replace(")", "")
    model_path = os.path.join("saved_models", f"{safe_name}.joblib")
    
    if os.path.exists(model_path):
        print(f"Loading pre-trained {name}...")
        saved_data = joblib.load(model_path)
        trained_model = saved_data['model']
        train_time = saved_data['train_time']
    else:
        print(f"Training {name}...")
        start_time = time.time()
        
        # Train the model (MLP is checkpointed; delete its .ckpt file to restart from scratch)
        if isinstance(model, MLPClassifier):
            ckpt_path = model_path + ".ckpt"
            trained_model, train_time = fit_mlp_checkpointed(model, X_train_vec, y_train, ckpt_path, save_every=1)
        else:
            trained_model = model.fit(X_train_vec, y_train)
            train_time = time.time() - start_time
        
        # Save both the model and its training time
        joblib.dump({'model': trained_model, 'train_time': train_time}, model_path)
        if isinstance(model, MLPClassifier) and os.path.exists(ckpt_path):
            os.remove(ckpt_path)  # training finished, checkpoint no longer needed

    # Update the dictionary so it holds the fitted model
    models[name] = trained_model
    
    # Predict on test set
    y_pred = trained_model.predict(X_test_vec)
    
    # Calculate metrics
    # using average='weighted' to account for class imbalance
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Training Time (s)": round(train_time, 2)
    })

# ── Soft Voting Ensemble (F1-weighted) ──────────────────────────────────
# Collect each model's F1 score as its vote weight
model_f1_weights = {r["Model"]: r["F1-Score"] for r in results}

# Determine the ordered label set from the training data
label_classes = sorted(y_train.unique())
label_to_idx  = {lbl: i for i, lbl in enumerate(label_classes)}

def softmax(x):
    """Numerically-stable row-wise softmax."""
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def get_proba_matrix(model, X):
    """
    Return a (n_samples, n_classes) probability matrix.
    Models that expose predict_proba use it directly.
    LinearSVC (and any other model with only decision_function) uses
    softmax over the raw decision scores as a calibrated proxy.
    """
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    else:
        scores = model.decision_function(X)          # (n_samples, n_classes)
        if scores.ndim == 1:                          # binary edge-case guard
            scores = np.column_stack([-scores, scores])
        return softmax(scores)

# Accumulate weighted probability matrices
weighted_proba_sum = np.zeros((X_test_vec.shape[0], len(label_classes)))

for name, model in models.items():
    w = model_f1_weights[name]
    proba = get_proba_matrix(model, X_test_vec)      # (n_samples, n_classes)

    # Align columns: sklearn may order classes differently per model
    model_classes = (model.classes_ if hasattr(model, "classes_")
                     else label_classes)
    aligned = np.zeros_like(weighted_proba_sum)
    for col_idx, cls in enumerate(model_classes):
        if cls in label_to_idx:
            aligned[:, label_to_idx[cls]] += proba[:, col_idx]

    weighted_proba_sum += w * aligned

# Final prediction: class with highest weighted probability
consensus_preds = np.array([label_classes[i]
                             for i in weighted_proba_sum.argmax(axis=1)])

# Evaluate soft-voting ensemble
acc_c = accuracy_score(y_test, consensus_preds)
precision_c, recall_c, f1_c, _ = precision_recall_fscore_support(
    y_test,
    consensus_preds,
    average='weighted',
    zero_division=0
)

results.append({
    "Model": "Ensemble Classifier",
    "Accuracy": acc_c,
    "Precision": precision_c,
    "Recall": recall_c,
    "F1-Score": f1_c,
    "Training Time (s)": 0.0
})

# Create a DataFrame for the results
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
display(results_df)


### 5. Visualizing the Comparison Analysis

In [ ]:
from sklearn.metrics import matthews_corrcoef, cohen_kappa_score, confusion_matrix
from itertools import combinations
from matplotlib.ticker import LogLocator, ScalarFormatter

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: F1-Score Comparison
sns.barplot(data=results_df, x="F1-Score", y="Model",hue=y,legend=False,ax=axes[0], palette="viridis")
axes[0].set_title("Model Comparison: Weighted F1-Score", fontsize=14)
axes[0].set_xlabel("F1-Score")
axes[0].set_ylabel("")

# Plot 2: Training Time Comparison
sns.barplot(data=results_df,x="Training Time (s)",y="Model",hue=y,legend=False,ax=axes[1],palette="rocket")

axes[1].set_xscale("log")
axes[1].xaxis.set_major_locator(LogLocator(base=10))
axes[1].xaxis.set_major_formatter(ScalarFormatter())
axes[1].set_title("Model Comparison: Training Time (Seconds)", fontsize=14)
axes[1].set_xlabel("Training Time (s, log scale)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

advanced_results = []

# Helper: get predictions for any model name (including consensus)
def get_predictions(name):
    if name == "Ensemble Classifier":
        return consensus_preds
    return models[name].predict(X_test_vec)

model_names = list(models.keys())
all_model_names = model_names + ["Ensemble Classifier"]
for name in all_model_names:
    y_pred = get_predictions(name)
    
    # Calculate advanced metrics
    macro_f1 = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)[2]
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    
    advanced_results.append({
        "Model": name,
        "Macro F1-Score": macro_f1,
        "MCC": mcc,
        "Cohen's Kappa": kappa
    })

advanced_results_df = pd.DataFrame(advanced_results).sort_values(by="MCC", ascending=False)
display(advanced_results_df)

predictions_dict = {}

for name in model_names:
    predictions_dict[name] = models[name].predict(X_test_vec)

prediction_matrix = np.column_stack([predictions_dict[name] for name in model_names])

similarity_scores = {}

for model1, model2 in combinations(model_names, 2):
    pred1 = predictions_dict[model1]
    pred2 = predictions_dict[model2]
    
    similarity = (pred1 == pred2).mean()
    similarity_scores[(model1, model2)] = similarity

most_similar_pair = max(similarity_scores, key=similarity_scores.get)
most_similar_score = similarity_scores[most_similar_pair]

most_differing_pair = min(similarity_scores, key=similarity_scores.get)
most_differing_score = similarity_scores[most_differing_pair]

# Print results
print(f"\nΤα δύο πιο παρόμοια μοντέλα ήταν τα {most_similar_pair[0]} και {most_similar_pair[1]}, με ποσοστό συμφωνίας {most_similar_score:.2%}.")
print(f"\nΤα δύο λιγότερο παρόμοια μοντέλα ήταν τα {most_differing_pair[0]} και {most_differing_pair[1]}, με ποσοστό συμφωνίας {most_differing_score:.2%}.")

# Plot Confusion Matrix for the best performing model (e.g., Random Forest)
best_model_name = advanced_results_df.iloc[0]['Model']
y_pred_best = get_predictions(best_model_name)
label_classes = sorted(y_test.unique())   # use true label set — works for all models incl. Consensus

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_best, labels=label_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_classes, yticklabels=label_classes)
plt.title(f"Confusion Matrix for best model: {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("True Label")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Re-create the exact test indices to map predictions back to the original text and datasets
_, test_indices, _, _ = train_test_split(df.index, y, test_size=0.2, random_state=42, stratify=y)
df_test = df.loc[test_indices].copy()

# 2. Identify the target datasets
target_datasets = ["MultiFC", "PUBHEALTH","FEVER"]
dataset_results = []

# 3. Iterate over each dataset to evaluate the pre-trained models
for dataset_name in target_datasets:
    # Create a boolean mask for the current dataset within the test dataframe
    mask = (df_test['dataset'] == dataset_name).values
    
    if not mask.any():
        print(f"No samples found for {dataset_name} in the test set. Skipping.")
        continue
        
    # Filter the vectorized features and the true labels using the mask
    X_test_subset = X_test_vec[mask]
    y_test_subset = df_test.iloc[mask]['label']
    
    all_eval_names = list(models.keys()) + ["Ensemble Classifier"]
    for model_name in all_eval_names:
        # Predict purely on this dataset's subset
        if model_name == "Ensemble Classifier":
            # Slice consensus_preds to the current dataset subset using the mask
            y_pred_subset = consensus_preds[mask]
        else:
            y_pred_subset = models[model_name].predict(X_test_subset)
        
        # Calculate metrics (using average='weighted' to handle potential class imbalances per dataset)
        acc = accuracy_score(y_test_subset, y_pred_subset)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test_subset, y_pred_subset, average='weighted', zero_division=0
        )
        
        dataset_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "Test Samples": mask.sum()
        })

# 4. Compile and display the results
dataset_results_df = pd.DataFrame(dataset_results)
print("Evaluation Metrics Broken Down by Dataset:")
display(dataset_results_df.sort_values(by=["Dataset", "F1-Score"], ascending=[True, False]))

# 5. Visualize the differences across datasets
plt.figure(figsize=(14, 6))
sns.barplot(
    data=dataset_results_df, 
    x="Model", 
    y="F1-Score", 
    hue="Dataset", 
    palette="mako"
)
plt.title("Model F1-Score Comparison Across Individual Datasets", fontsize=14)
plt.xlabel("Model")
plt.ylabel("Weighted F1-Score")
plt.xticks(rotation=45, ha='right')
plt.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
from dotenv import load_dotenv
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

nlp = spacy.load("en_core_web_sm")
ranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
nli_model = pipeline(
    "text-classification", 
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=0 if torch.cuda.is_available() else -1,
    token=hf_token
)

def chunk_text(text, sentences_per_chunk=6, overlap=2):
    sentences = re.split(r'(?<=[.!?]) +', text.strip())
    chunks = []
    step = max(1, sentences_per_chunk - overlap)
    for i in range(0, len(sentences), step):
        chunk = ' '.join(sentences[i:i + sentences_per_chunk])
        if len(chunk.split()) > 10:
            chunks.append(chunk)
    return chunks

def extract_search_query(claim_text):
    doc = nlp(claim_text)
    keywords = [token.text for token in doc if not token.is_stop or token.dep_ == 'neg']
    query = " ".join(keywords)
    return query if query.strip() else claim_text

def search_wikipedia(query, max_results=2):
    results_data = []
    try:
        search_results = wikipedia.search(query, results=max_results)
        for title in search_results:
            try:
                page = wikipedia.page(title, auto_suggest=False)
                chunks = chunk_text(page.content)
                for chunk in chunks:
                    results_data.append({
                        "source": "Wikipedia", "title": title, 
                        "url": page.url, "text": chunk
                    })
            except (wikipedia.exceptions.DisambiguationError, wikipedia.exceptions.PageError):
                continue
    except Exception:
        pass
    return results_data

def search_duckduckgo(query, max_results=3):
    results_data = []
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=max_results)
            for r in results:
                url = r.get("href")
                title = r.get("title")
                downloaded = trafilatura.fetch_url(url)
                if downloaded:
                    article_text = trafilatura.extract(downloaded)
                    if article_text:
                        chunks = chunk_text(article_text)
                        for chunk in chunks:
                            results_data.append({
                                "source": "DuckDuckGo", "title": title, 
                                "url": url, "text": chunk
                            })
    except Exception as e:
        print(f"DDG search error: {e}")
    return results_data

def rank_results(claim, results, threshold=-10):
    if not results: return []
    pairs = [[claim, res["text"]] for res in results]
    scores = ranker_model.predict(pairs)
    
    for i, score in enumerate(scores):
        results[i]["relevance_score"] = float(score)
        
    ranked = sorted(results, key=lambda x: x["relevance_score"], reverse=True)
    
    seen_urls, unique_results = set(), []
    for res in ranked:
        if res["relevance_score"] >= threshold and res["url"] not in seen_urls:
            seen_urls.add(res["url"])
            unique_results.append(res)
    return unique_results

random_sample = df.sample(1).iloc[0]

claim_to_check = random_sample['claim']
original_label = random_sample['label']

print(f"Selected Claim: {claim_to_check}\n")
print(f"Original Label: {original_label}\n")
# 2. Retrieve live context using advanced pipeline
search_query = extract_search_query(claim_to_check)
print(f"Searching Wikipedia and live web for: '{search_query}'...")

raw_results = search_wikipedia(search_query) + search_duckduckgo(search_query)
print(f"Collected {len(raw_results)} chunks. Ranking evidence...")

ranked_evidence = rank_results(claim_to_check, raw_results)

# Build retrieved_evidence string from the top 3 highly-ranked chunks
retrieved_evidence = ""
for i, res in enumerate(ranked_evidence[:3]):
    retrieved_evidence += f"{res['text']} "
    print(f"  [{i+1}] Source: {res['source']} - {res['title']} (Score: {res['relevance_score']:.2f})")

if not retrieved_evidence.strip():
    retrieved_evidence = "Could not retrieve highly relevant web search data."

print("-" * 60)

# 3. Run DeBERTa NLI Analysis
result = nli_model({"text": retrieved_evidence, "text_pair": claim_to_check})

label_mapping = {
    "entailment": "SUPPORTED",
    "contradiction": "REFUTED",
    "neutral": "NOT_ENOUGH_INFO"
}

predicted_label = result['label'].lower()
predicted_label = label_mapping.get(predicted_label, predicted_label)
confidence = result['score']
print(f"RAG CLASSIFICATION: {predicted_label}")
print(f"CONFIDENCE SCORE: {confidence:.2%}")

claim_vec = claim_vectorizer.transform([claim_to_check])
evidence_vec = evidence_vectorizer.transform([retrieved_evidence])
input_vec = hstack([claim_vec, evidence_vec]).tocsr()

In [ ]:
from dotenv import load_dotenv
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

llm_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 

print(f"\nLoading LLM...")
llm_pipeline = pipeline(
    "text-generation", 
    model=llm_model_id, 
    # Use float16 to save memory if a GPU is available
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    token=hf_token
)

# 2. Define the strict system behavior
system_prompt = (
    "You are a strict fact-checking assistant. Your task is to classify claims into EXACTLY "
    "one of these labels: SUPPORTED, REFUTED, or UNKNOWN."
    "Do not provide explanations, just output the label."
)

# 3. Create Prompts using the model's chat template
# Scenario A: Internal Memory (Zero-Shot)
prompt_memory = (
    f"<|system|>\n{system_prompt}</s>\n"
    f"<|user|>\nClaim: {claim_to_check}</s>\n"
    f"<|assistant|>\nLabel:"
)

# Scenario B: With RAG Context
prompt_rag = (
    f"<|system|>\n{system_prompt} You must base your decision ONLY on the provided evidence.</s>\n"
    f"<|user|>\nEvidence: {retrieved_evidence}\n\nClaim: {claim_to_check}</s>\n"
    f"<|assistant|>\nLabel:"
)

# 4. Helper function to generate and extract the label
def get_llm_prediction(prompt_text):
    output = llm_pipeline(
        prompt_text, 
        max_new_tokens=20,
        return_full_text=False,
        do_sample=False,      
        temperature=None,
        top_p=None
    )
    
    response = output[0]['generated_text'].strip()
    
    valid_labels = ['SUPPORTED', 'REFUTED', 'UNKNOWN']
    
    if "unknown" in response.lower():
        return "NOT_ENOUGH_INFO"
        
    for label in valid_labels:
        if label.lower() in response.lower():
            return label
    
    # Fallback if the LLM gets wordy or fails to follow instructions
    return f"UNKNOWN_FORMAT ({response})"

# 5. Run the predictions
print("Running LLM (Internal Memory)...")
pred_memory = get_llm_prediction(prompt_memory)

print("Running LLM (With RAG Context)...")
pred_rag = get_llm_prediction(prompt_rag)

original_label = original_label.upper()
# 6. Print the unified final results
print("=" * 60)
print(f"CLAIM: {claim_to_check}")
print(f"DATASET LABEL: {original_label}")
print("-" * 60)
print(f"{'Algorithm':<25} | {'Predicted Label'}")
print("-" * 60)

# Loop through traditional trained models and collect weighted probabilities
# Reuse the same helpers (softmax, get_proba_matrix, label_classes, label_to_idx)
# and the model F1 weights already stored in results_df.
model_f1_lookup = dict(zip(results_df["Model"], results_df["F1-Score"]))

weighted_proba_single = np.zeros(len(label_classes))

for name, model in models.items():
    prediction = model.predict(input_vec)[0]
    print(f"{name:<25} | {prediction:}")

    w = model_f1_lookup.get(name, 1.0)
    proba = get_proba_matrix(model, input_vec)[0]          # 1-D array

    model_classes = (model.classes_ if hasattr(model, "classes_")
                     else label_classes)
    for col_idx, cls in enumerate(model_classes):
        if cls in label_to_idx:
            weighted_proba_single[label_to_idx[cls]] += w * proba[col_idx]

consensus_pred = label_classes[weighted_proba_single.argmax()]

print("-" * 60)
print(f"{'Consensus (Soft Vote, F1-weighted)':<25} | {consensus_pred}")
print("-" * 60)
# Print the DeBERTa Pipeline
print(f"{'RAG (DeBERTa NLI)':<25} | {label_mapping.get(predicted_label, predicted_label)}")
# Print the new LLM methods
print(f"{'LLM (Internal Memory)':<25} | {pred_memory}")
print(f"{'LLM (With RAG Context)':<25} | {pred_rag}")
print("=" * 60)

In [ ]:
# 1. Re-create the exact test indices to map predictions back to the original text
_, test_indices, _, _ = train_test_split(df.index, y, test_size=0.2, random_state=42, stratify=y)
df_test = df.loc[test_indices].copy()

# 2. Gather predictions from all models
model_names = list(models.keys())
for name, model in models.items():
    df_test[name] = model.predict(X_test_vec)

# 3. Calculate consensus metrics per row
# How many models got it exactly right?
df_test['Num_Correct'] = (df_test[model_names].values == df_test[['label']].values).sum(axis=1)
# How many unique predictions were made across the 7 models? (Higher = more mixed)
df_test['Num_Unique_Preds'] = df_test[model_names].nunique(axis=1)

# --- SCENARIO A: Everyone got it right ---
sample_correct = df_test[df_test['Num_Correct'] == len(models)].sample(1).iloc[0]

# --- SCENARIO B: Most mixed classifications (highest disagreement) ---
# Find the row with the maximum number of unique predictions
sample_mixed = df_test.loc[df_test['Num_Unique_Preds'].idxmax()]

# --- SCENARIO C: Everyone agreed on the WRONG label ---
# 0 correct predictions, but exactly 1 unique prediction across all 7 models
sample_wrong_consensus = df_test[(df_test['Num_Correct'] == 0) & (df_test['Num_Unique_Preds'] == 1)].sample(1).iloc[0]

samples_to_test = {
    "1. Perfect Agreement (Correct)": sample_correct,
    "2. Highly Mixed": sample_mixed,
    "3. Perfect Agreement (Incorrect)": sample_wrong_consensus
}

for desc, sample in samples_to_test.items():
    print(f"--- {desc} ---")
    print(f"Claim: {sample['claim']}")
    print(f"True Label: {sample['label']}")
    print("Model Predictions:")
    for model_name in model_names:
        print(f"  {model_name}: {sample[model_name]}")

In [ ]:
def evaluate_llm_pipeline(claim, true_label):
    print(f"{'='*80}\nEvaluating Claim: {claim}\nTrue Label: {true_label}\n{'-'*80}")
    
    # 1. Retrieve Context
    search_query = extract_search_query(claim)
    raw_results = search_wikipedia(search_query) + search_duckduckgo(search_query)
    ranked_evidence = rank_results(claim, raw_results)
    
    retrieved_evidence = " ".join([res['text'] for res in ranked_evidence[:3]])
    if not retrieved_evidence.strip():
        retrieved_evidence = "Could not retrieve highly relevant web search data."
        
    # 2. DeBERTa NLI Pipeline
    nli_result = nli_model({"text": retrieved_evidence, "text_pair": claim})
    nli_pred = label_mapping.get(nli_result['label'].lower(), nli_result['label'].lower())
    
    # 3. Build Prompts
    prompt_mem = f"<|system|>\n{system_prompt}</s>\n<|user|>\nClaim: {claim}</s>\n<|assistant|>\nLabel:"
    prompt_context = f"<|system|>\n{system_prompt} You must base your decision ONLY on the provided evidence.</s>\n<|user|>\nEvidence: {retrieved_evidence}\n\nClaim: {claim}</s>\n<|assistant|>\nLabel:"
    
    # 4. LLM Generation
    pred_mem = get_llm_prediction(prompt_mem)
    pred_context = get_llm_prediction(prompt_context)
    
    # 5. Output Results
    print(f"{'RAG (DeBERTa NLI)':<25} | {nli_pred} (Conf: {nli_result['score']:.2%})")
    print(f"{'LLM (Internal Memory)':<25} | {pred_mem}")
    print(f"{'LLM (With RAG Context)':<25} | {pred_context}")
    print(f"{'='*80}")

# Run the evaluation on our 3 targeted samples
for desc, sample in samples_to_test.items():
    print(f"\n{desc}")
    evaluate_llm_pipeline(sample['claim'], sample['label'])